# Task 1 - LoRA fine-tuning of a small model (Qwen2.5-0.5B) for text -> JSON

**Task:** turn a plain-English meeting request into a fixed JSON schema.

**Where Groq fits (important):** Groq only serves models for inference - you cannot fine-tune on it.
So the pipeline is:
1. **Groq (Llama-3.3-70B)** writes realistic, messy natural-language requests for programmatically generated JSON labels (labels are exact by construction).
2. **Colab T4 GPU** does the LoRA fine-tuning of Qwen2.5-0.5B-Instruct.
3. We compare base vs fine-tuned on 100 held-out examples.

**Before you run:**
- Runtime > Change runtime type > **T4 GPU**
- Get a free key at https://console.groq.com/keys, then in Colab click the key icon (Secrets), add `GROQ_API_KEY`, and turn on "Notebook access".

In [15]:
!pip install -q -U transformers peft accelerate datasets groq pandas
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

torch 2.14.1+cu130 | transformers 5.23.0 | peft 0.21.3
GPU: Tesla T4


In [16]:
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key: ")

from groq import Groq
client = Groq()
# Updated to a currently active production model on Groq
PARA_MODEL = "openai/gpt-oss-120b"   # if you hit rate limits, try "llama-3.1-8b-instant"
r = client.chat.completions.create(model=PARA_MODEL, max_tokens=5,
                                   messages=[{"role": "user", "content": "Say OK"}])
print("Groq works ->", r.choices[0].message.content)

Groq works -> 


## Step 1 - Build the dataset
We first generate 600 random JSON "specs" (the gold labels), then ask Groq to write a natural request for each one.
If Groq's text drops a required fact (attendee name, month, day) we fall back to a template sentence, so labels are never wrong because of the paraphraser.
Split: 450 train / 50 validation / **100 test** (test is never used for training or tuning).

In [17]:
import random, json, time, datetime, os
random.seed(42)
os.makedirs("/content/data", exist_ok=True)

ACTIONS = ["schedule", "cancel", "reschedule"]
TITLES = ["budget review", "design sync", "project kickoff", "sprint planning", "client demo",
          "quarterly planning", "onboarding session", "code review", "team retro", "roadmap discussion",
          "vendor call", "hiring debrief", "security audit", "product launch prep", "performance review"]
NAMES = ["Priya", "Rahul", "Anita", "Karthik", "Sneha", "Vikram", "Meera", "Arjun",
         "Divya", "Imran", "Neha", "Suresh", "Kavya", "Rohan", "Fatima", "Sanjay"]
DURATIONS = [15, 30, 45, 60, 90, 120]
PRIORITIES = ["low", "medium", "high"]

def make_spec():
    return {
        "action": random.choice(ACTIONS),
        "title": random.choice(TITLES),
        "date": f"2026-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        "time": f"{random.randint(8,18):02d}:{random.choice(['00','15','30','45'])}",
        "duration_min": random.choice(DURATIONS),
        "attendees": random.sample(NAMES, random.choice([1, 1, 2, 2, 3])),
        "priority": random.choice(PRIORITIES),
    }

N_TOTAL = 600
specs, seen = [], set()
while len(specs) < N_TOTAL:
    s = make_spec()
    k = json.dumps(s)
    if k not in seen:
        seen.add(k)
        specs.append(s)

def fallback_text(s):
    d = datetime.date.fromisoformat(s["date"])
    h, m = map(int, s["time"].split(":"))
    ampm = "am" if h < 12 else "pm"
    t = f"{(h % 12) or 12}:{m:02d}{ampm}"
    prio = {"high": "This is urgent.", "medium": "Normal priority.", "low": "No rush."}[s["priority"]]
    verb = {"schedule": "Please schedule a", "cancel": "Please cancel the", "reschedule": "Please reschedule the"}[s["action"]]
    who = " and ".join(s["attendees"])
    return f"{verb} {s['title']} with {who} on {d.strftime('%B')} {d.day}, {d.year} at {t} for {s['duration_min']} minutes. {prio}"

PARA_SYS = (
    "You write realistic messages that a busy person might type to an assistant. "
    "For each JSON spec you receive, write ONE natural request that conveys exactly the same facts. Rules: "
    "mention the action (schedule / cancel / reschedule) naturally; the meeting title; "
    "the date with the month written as a WORD, the day number as digits, and the year 2026; "
    "the time in any natural format (3pm, 15:30, 3:30 in the afternoon); "
    "the duration in any natural phrasing (an hour, 90 minutes, half an hour); "
    "every attendee by first name; and the priority through natural wording "
    "(high = urgent / ASAP / top priority; low = no rush / whenever / low priority; medium = normal / standard / regular). "
    "Vary the style: some terse, some polite, some chatty. Never copy the JSON keys into the request. "
    'Return ONLY a JSON object: {"requests": ["...", "..."]} with exactly one string per spec, in the same order.'
)

def paraphrase_batch(batch, tries=5):
    for attempt in range(tries):
        try:
            resp = client.chat.completions.create(
                model=PARA_MODEL, temperature=0.9,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": PARA_SYS},
                          {"role": "user", "content": json.dumps(batch)}])
            texts = json.loads(resp.choices[0].message.content)["requests"]
            if len(texts) == len(batch):
                return texts
        except Exception as e:
            wait = 15 * (attempt + 1)
            print("retry:", str(e)[:120], "| waiting", wait, "s")
            time.sleep(wait)
    return [None] * len(batch)

def text_ok(text, s):
    if not isinstance(text, str):
        return False
    t = text.lower()
    d = datetime.date.fromisoformat(s["date"])
    return (all(n.lower() in t for n in s["attendees"])
            and d.strftime("%B").lower() in t and str(d.day) in t)

CACHE = "/content/data/all.json"
if os.path.exists(CACHE):
    rows = json.load(open(CACHE))
    print("loaded cached dataset:", len(rows))
else:
    rows, n_fallback, BATCH = [], 0, 10
    for i in range(0, len(specs), BATCH):
        batch = specs[i:i + BATCH]
        texts = paraphrase_batch(batch)
        for s, t in zip(batch, texts):
            if not text_ok(t, s):
                t = fallback_text(s)
                n_fallback += 1
            rows.append({"request": t.strip(), "target": s})
        if (i // BATCH) % 5 == 0:
            print(f"{len(rows)}/{len(specs)} done | template fallbacks so far: {n_fallback}")
        time.sleep(2)
    json.dump(rows, open(CACHE, "w"))
    print("total template fallbacks:", n_fallback)

train, val, test = rows[:450], rows[450:500], rows[500:]
print(len(train), len(val), len(test))
for r in test[:3]:
    print("\nREQUEST:", r["request"], "\nGOLD   :", json.dumps(r["target"]))

loaded cached dataset: 600
450 50 100

REQUEST: Hey, can we reschedule the product launch prep with Priya and Neha to May 5, 2026 at 3 pm for an hour? No rush. 
GOLD   : {"action": "reschedule", "title": "product launch prep", "date": "2026-05-05", "time": "15:00", "duration_min": 60, "attendees": ["Priya", "Neha"], "priority": "low"}

REQUEST: Please reschedule the vendor call with Vikram to March 25, 2026 at 4:15 pm for an hour, ASAP. 
GOLD   : {"action": "reschedule", "title": "vendor call", "date": "2026-03-25", "time": "16:15", "duration_min": 60, "attendees": ["Vikram"], "priority": "high"}

REQUEST: Could we move the vendor call with Sanjay to March 24, 2026 at 3:30 pm for fifteen minutes? Whenever works. 
GOLD   : {"action": "reschedule", "title": "vendor call", "date": "2026-03-24", "time": "15:30", "duration_min": 15, "attendees": ["Sanjay"], "priority": "low"}


## Step 2 - Prompts, tokenisation, LoRA

**LoRA settings and why (copy this reasoning into your report):**

| Setting | Value | Why |
|---|---|---|
| Rank `r` | 16 | Task is narrow (fixed schema); r=8-16 is enough. 16 gives headroom for the date/time reasoning without overfitting 450 examples. |
| `lora_alpha` | 32 | Scaling = alpha/r = 2, the common default ratio; keeps update magnitude stable. |
| Target modules | q,k,v,o + gate,up,down (all linear layers) | Adapting attention **and** MLP gives better quality than attention-only (QLoRA paper); on a 0.5B model the extra params are tiny. |
| Dropout | 0.05 | Light regularisation for a small dataset. |
| Learning rate | 2e-4 | Standard for LoRA (10x higher than full FT because only small adapter matrices train). Cosine schedule + 5% warmup. |
| Epochs | 3 | Loss plateaus by ~3 epochs on 450 examples; more starts to overfit (watch validation loss). |
| Batch | 8 x 2 accumulation = 16 | Fits a T4 comfortably. |
| Loss masking | only on the JSON answer | Model is not trained to reproduce the prompt. |

In [20]:
# Upgrade torchao to resolve the compatibility error with peft
!pip install -q "torchao>=0.16.0"

import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from datasets import Dataset

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"     # alternatives: "Qwen/Qwen2.5-1.5B-Instruct" (still <= 1.5B)
tok = AutoTokenizer.from_pretrained(MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
model.to("cuda")

# Short prompt used for fine-tuning (the model must LEARN the schema)
SYSTEM_SHORT = "Convert the user's meeting request into JSON."
# Long prompt that spells out the schema - gives the BASE model its best chance in the comparison
SYSTEM_SCHEMA = (
    "You convert meeting requests into JSON. Output ONLY a JSON object with exactly these keys: "
    'action ("schedule"|"cancel"|"reschedule"), title (lowercase string), date (YYYY-MM-DD, year 2026), '
    'time (24-hour HH:MM), duration_min (integer minutes), attendees (list of first names), '
    'priority ("low"|"medium"|"high"). No explanations, no markdown.'
)

def build_prompt(system, request):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": request}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

MAX_LEN = 320
def encode(ex):
    prompt = build_prompt(SYSTEM_SHORT, ex["request"])
    answer = json.dumps(ex["target"]) + "<|im_end|>"
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p + a)[:MAX_LEN]
    labels = ([-100] * len(p) + a)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_ds = Dataset.from_list([encode(x) for x in train])
val_ds = Dataset.from_list([encode(x) for x in val])
print("max sequence length in train:", max(len(x) for x in train_ds["input_ids"]))

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

max sequence length in train: 148


W1118 10:31:03.101000 417 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


## Step 3 - Train
Base weights stay in fp32 with fp16 autocast (the T4 has no bf16). 0.5B in fp32 is only ~2 GB, so this is safe and avoids the "unscale FP16 gradients" error.
Training takes a few minutes on a T4.

In [22]:
args = TrainingArguments(
    output_dir="/content/lora-out",
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3, learning_rate=2e-4,
    lr_scheduler_type="cosine", warmup_steps=10, weight_decay=0.0,
    fp16=True, logging_steps=10,
    eval_strategy="epoch", save_strategy="no",
    report_to="none", remove_unused_columns=False,
)
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator)

torch.cuda.reset_peak_memory_stats()
trainer.train()
print(f"\nPeak GPU memory during LoRA training: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Epoch,Training Loss,Validation Loss
1,0.023688,0.002778
2,0.003209,0.001461
3,0.001390,0.000793



Peak GPU memory during LoRA training: 7.51 GB


## Step 4 - Evaluate: base vs fine-tuned on the 100 held-out test examples
Three runs, all greedy decoding:
- **Base + short prompt** (same prompt the fine-tuned model sees)
- **Base + schema prompt** (best-effort prompting baseline)
- **Fine-tuned + short prompt**

Metrics: strict JSON validity, lenient JSON validity (extracts `{...}` from the text), **exact match** (all 7 fields correct), and per-field accuracy.

In [23]:
import pandas as pd
tok.padding_side = "left"

@torch.no_grad()
def generate_batch(system, requests, bs=16, max_new=150):
    outs = []
    for i in range(0, len(requests), bs):
        prompts = [build_prompt(system, r) for r in requests[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for g in gen[:, enc["input_ids"].shape[1]:]:
            outs.append(tok.decode(g, skip_special_tokens=True).strip())
    return outs

FIELDS = ["action", "title", "date", "time", "duration_min", "attendees", "priority"]

def extract_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e < s:
        return None
    try:
        obj = json.loads(text[s:e + 1])
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def strict_json(text):
    try:
        return isinstance(json.loads(text), dict)
    except Exception:
        return False

def norm_val(f, v):
    try:
        if f == "attendees":
            return sorted(str(x).strip().lower() for x in v)
        if f == "duration_min":
            return int(v)
        return str(v).strip().lower()
    except Exception:
        return None

def score(preds, golds):
    n = len(golds)
    parsed = [extract_json(p) for p in preds]
    valid = sum(p is not None for p in parsed)
    strict = sum(strict_json(p) for p in preds)
    exact, per = 0, {f: 0 for f in FIELDS}
    for p, g in zip(parsed, golds):
        if p is None:
            continue
        all_ok = True
        for f in FIELDS:
            good = norm_val(f, p.get(f)) == norm_val(f, g[f])
            per[f] += int(good)
            all_ok = all_ok and good
        exact += int(all_ok)
    res = {"strict_json_%": 100 * strict / n, "valid_json_%": 100 * valid / n,
           "exact_match_%": 100 * exact / n,
           "field_acc_%": 100 * sum(per.values()) / (n * len(FIELDS))}
    res.update({f"acc_{f}_%": 100 * per[f] / n for f in FIELDS})
    return res

test_req = [r["request"] for r in test]
test_gold = [r["target"] for r in test]
model.eval()
model.config.use_cache = True

with model.disable_adapter():                      # adapter OFF = original base model
    out_base_short = generate_batch(SYSTEM_SHORT, test_req)
    out_base_schema = generate_batch(SYSTEM_SCHEMA, test_req)
out_ft = generate_batch(SYSTEM_SHORT, test_req)    # adapter ON

results = pd.DataFrame({
    "Base (short prompt)": score(out_base_short, test_gold),
    "Base (schema prompt)": score(out_base_schema, test_gold),
    "Fine-tuned (LoRA)": score(out_ft, test_gold),
}).round(1)
results

,Base (short prompt),Base (schema prompt),Fine-tuned (LoRA)
strict_json_%,2.0,67.0,100.0
valid_json_%,98.0,100.0,100.0
exact_match_%,0.0,22.0,94.0
field_acc_%,1.9,81.6,99.1
acc_action_%,0.0,95.0,100.0
acc_title_%,0.0,78.0,100.0
acc_date_%,5.0,100.0,100.0
acc_time_%,0.0,60.0,100.0
acc_duration_min_%,0.0,75.0,95.0
acc_attendees_%,2.0,100.0,100.0


In [24]:
# Look at raw outputs (useful for the report)
for i in range(3):
    print("REQUEST   :", test_req[i])
    print("GOLD      :", json.dumps(test_gold[i]))
    print("BASE      :", out_base_schema[i][:200].replace("\n", " "))
    print("FINE-TUNED:", out_ft[i])
    print("-" * 80)

REQUEST   : Hey, can we reschedule the product launch prep with Priya and Neha to May 5, 2026 at 3 pm for an hour? No rush.
GOLD      : {"action": "reschedule", "title": "product launch prep", "date": "2026-05-05", "time": "15:00", "duration_min": 60, "attendees": ["Priya", "Neha"], "priority": "low"}
BASE      : ```json {   "action": "reschedule",   "title": "Product Launch Prep",   "date": "2026-05-05",   "time": "13:00",   "duration_min": 60,   "attendees": ["Priya", "Neha"],   "priority": "high" } ```
FINE-TUNED: {"action": "reschedule", "title": "product launch prep", "date": "2026-05-05", "time": "15:00", "duration_min": 60, "attendees": ["Priya", "Neha"], "priority": "low"}
--------------------------------------------------------------------------------
REQUEST   : Please reschedule the vendor call with Vikram to March 25, 2026 at 4:15 pm for an hour, ASAP.
GOLD      : {"action": "reschedule", "title": "vendor call", "date": "2026-03-25", "time": "16:15", "duration_min": 60, "at

## Step 5 - Failure analysis (pick 3 for your report)
The cell prints which fields fail most and shows concrete failed cases. For each of your 3 examples, explain the cause. Typical causes to check:
- **Priority = "medium"**: it is expressed only by weak words ("normal", "standard") or by nothing, so it is genuinely ambiguous.
- **Time conversion**: "half past two in the afternoon" or "3 in the evening" needs 12h -> 24h reasoning, and a 0.5B model has weak arithmetic.
- **Date**: day numbers written as words ("the fourteenth"), or the model copying a wrong digit.
- **Duration**: "an hour and a half", "a quarter of an hour".
- **Label noise**: Groq's paraphrase kept the names/month/day but changed something else (check by reading the request yourself).

In [25]:
from collections import Counter
fails = []
for req, g, raw in zip(test_req, test_gold, out_ft):
    p = extract_json(raw)
    wrong = [f for f in FIELDS if p is None or norm_val(f, p.get(f)) != norm_val(f, g[f])]
    if wrong:
        fails.append((req, g, raw, wrong))

print(f"{len(fails)} failures out of {len(test_req)} test examples")
print("Errors per field:", dict(Counter(f for *_, w in fails for f in w)))
for req, g, raw, wrong in fails[:8]:
    print("\nREQUEST :", req)
    print("GOLD    :", json.dumps(g))
    print("PREDICTED:", raw)
    print("WRONG FIELDS:", wrong)

6 failures out of 100 test examples
Errors per field: {'duration_min': 5, 'priority': 1}

REQUEST : Please cancel the quarterly planning with Kavya on October 21, 2026 at 9:30 am, regular priority.
GOLD    : {"action": "cancel", "title": "quarterly planning", "date": "2026-10-21", "time": "09:30", "duration_min": 60, "attendees": ["Kavya"], "priority": "medium"}
PREDICTED: {"action": "cancel", "title": "quarterly planning", "date": "2026-10-21", "time": "09:30", "duration_min": 45, "attendees": ["Kavya"], "priority": "medium"}
WRONG FIELDS: ['duration_min']

REQUEST : Cancel the code review with Arjun scheduled for August 20, 2026 at 8:15 am, low priority.
GOLD    : {"action": "cancel", "title": "code review", "date": "2026-08-20", "time": "08:15", "duration_min": 60, "attendees": ["Arjun"], "priority": "low"}
PREDICTED: {"action": "cancel", "title": "code review", "date": "2026-08-20", "time": "08:15", "duration_min": 45, "attendees": ["Arjun"], "priority": "low"}
WRONG FIELDS: ['dura

## Step 6 - Memory calculation for a 7B model (with working)

**Rule of thumb, mixed-precision AdamW:**
- fp16 weights: 2 B/param
- fp16 gradients: 2 B/param
- fp32 master weights: 4 B/param
- Adam moment m (fp32): 4 B/param, Adam moment v (fp32): 4 B/param
- Total = **16 B/param** (before activations)

**(a) Full fine-tuning:** 7e9 x 16 = **112 GB** + activations (~5-20 GB) -> ~120-130 GB (needs 2x A100-80GB or more).

**(b) LoRA (r=16, all linear layers):** frozen fp16 weights 7e9 x 2 = **14 GB**. Trainable params for a Llama-7B shape (32 layers, hidden 4096, MLP 11008):
per layer = 4 attention matrices x r x (4096+4096) + 3 MLP matrices x r x (4096+11008) = 78,080 x r; x 32 layers x r=16 = **~40M params**.
Their training state (fp32 weights 4 + grads 4 + Adam 8 = 16 B) = 40M x 16 = **~0.64 GB**. Activations ~3-5 GB.
Total ~ 14 + 0.64 + ~4 = **~19 GB** (fits a 24 GB card).

**(c) QLoRA:** base weights in 4-bit NF4 = 7e9 x 0.5 B = **3.5 GB** (+ ~0.2 GB quantisation constants with double quant) ~ 3.7 GB. LoRA state 0.64 GB. Activations ~2-3 GB with gradient checkpointing; paged optimiser absorbs spikes.
Total ~ 3.7 + 0.64 + ~2.5 = **~7 GB** (fits a free 16 GB T4, even a 12 GB card).

The next cell reproduces this arithmetic.

In [26]:
P = 7e9
GB = 1e9

# (a) Full fine-tuning, mixed precision AdamW
full_states = P * (2 + 2 + 4 + 4 + 4) / GB
act_full = 10          # assumption: ~10 GB activations (batch 1-4, seq 512, no checkpointing)
print(f"(a) Full FT : states {full_states:.0f} GB + activations ~{act_full} GB = ~{full_states + act_full:.0f} GB")

# (b) LoRA
layers, h, inter, r = 32, 4096, 11008, 16
lora_params = layers * (4 * r * (h + h) + 3 * r * (h + inter))
lora_state = lora_params * (4 + 4 + 8) / GB       # fp32 weights + grads + Adam m,v
frozen_fp16 = P * 2 / GB
act_lora = 4           # assumption
print(f"(b) LoRA    : trainable params {lora_params/1e6:.1f}M | frozen weights {frozen_fp16:.1f} GB "
      f"+ LoRA state {lora_state:.2f} GB + activations ~{act_lora} GB = ~{frozen_fp16 + lora_state + act_lora:.1f} GB")

# (c) QLoRA
nf4 = P * 0.5 / GB + 0.2                          # 4-bit weights + quantisation constants
act_q = 2.5            # assumption: with gradient checkpointing
print(f"(c) QLoRA   : 4-bit weights {nf4:.1f} GB + LoRA state {lora_state:.2f} GB "
      f"+ activations ~{act_q} GB = ~{nf4 + lora_state + act_q:.1f} GB")

print(f"\nMeasured peak for the 0.5B LoRA run above: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

(a) Full FT : states 112 GB + activations ~10 GB = ~122 GB
(b) LoRA    : trainable params 40.0M | frozen weights 14.0 GB + LoRA state 0.64 GB + activations ~4 GB = ~18.6 GB
(c) QLoRA   : 4-bit weights 3.7 GB + LoRA state 0.64 GB + activations ~2.5 GB = ~6.8 GB

Measured peak for the 0.5B LoRA run above: 7.51 GB


## Step 7 - Save the adapter and results

In [27]:
model.save_pretrained("/content/lora-meeting-json")
tok.save_pretrained("/content/lora-meeting-json")
results.to_csv("/content/results_base_vs_finetuned.csv")
with open("/content/failures.json", "w") as f:
    json.dump([{"request": a, "gold": b, "pred": c, "wrong_fields": d} for a, b, c, d in fails], f, indent=1)

!zip -qr /content/task1_outputs.zip /content/lora-meeting-json /content/results_base_vs_finetuned.csv /content/failures.json /content/data
from google.colab import files
files.download("/content/task1_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>